# Version 21.1 : CoxNet with Enriched Feature Engineering

**Amélioration vs V21** :
- ✨ Feature engineering **spécifique pour modèles linéaires**
- 🔗 **Interactions** entre features cliniques importantes
- 📐 **Transformations non-linéaires** (log, sqrt, carré)
- 📊 **Ratios cliniques** pertinents
- 🧬 **Interactions moléculaire × clinique**

**Objectif** : Améliorer le C-index de +0.02-0.04 grâce au feature engineering adapté

## 1. Setup & Imports

In [1]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Survival analysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# Set random seed
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

✓ Libraries imported


## 2. Data Loading

In [2]:
# Load data
import os
DATA_PATH = "C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(os.path.join(DATA_PATH, "X_train", "clinical_train.csv"))
target_train = pd.read_csv(os.path.join(DATA_PATH, "target_train.csv"))
clinical_test = pd.read_csv(os.path.join(DATA_PATH, "X_test", "clinical_test.csv"))
molecular_train = pd.read_csv(os.path.join(DATA_PATH, "X_train", "molecular_train.csv"))
molecular_test = pd.read_csv(os.path.join(DATA_PATH, "X_test", "molecular_test.csv"))

print(f"✓ Data loaded: {clinical_train.shape[0]} train patients")
print(f"  Test patients: {clinical_test.shape[0]}")

✓ Data loaded: 3323 train patients
  Test patients: 1193


## 3. Base Feature Engineering (same as before)

In [3]:
def create_cytogenetic_features(clinical_df):
    """Create cytogenetic features from CYTOGENETICS column"""
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    
    # Anomaly types
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    
    # Complexity
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    
    # Prognostic markers
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    """Create molecular features"""
    mol_features = pd.DataFrame({'ID': patient_ids})
    
    # Mutation counts
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    
    # VAF stats
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    
    # Effect types
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    
    # Top genes
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

print("✓ Base feature engineering functions defined")

✓ Base feature engineering functions defined


## 4. ENRICHED Feature Engineering (NEW!)

In [4]:
def add_enriched_features(X_clinical, X_molecular):
    """
    Add enriched features for linear models:
    - Non-linear transformations
    - Clinical ratios
    - Interactions between important features
    """
    X_enriched = pd.DataFrame(index=X_clinical.index)
    
    # === 1. NON-LINEAR TRANSFORMATIONS ===
    # Log transformations (add small constant to avoid log(0))
    for col in ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES']:
        if col in X_clinical.columns:
            X_enriched[f'{col}_log'] = np.log1p(X_clinical[col])  # log(1+x)
            X_enriched[f'{col}_sqrt'] = np.sqrt(X_clinical[col].clip(lower=0))
            X_enriched[f'{col}_squared'] = X_clinical[col] ** 2
    
    # === 2. CLINICAL RATIOS ===
    # Blood count ratios (clinically meaningful)
    if 'WBC' in X_clinical.columns and 'ANC' in X_clinical.columns:
        X_enriched['WBC_ANC_ratio'] = X_clinical['WBC'] / (X_clinical['ANC'] + 1e-5)
    
    if 'HB' in X_clinical.columns and 'PLT' in X_clinical.columns:
        X_enriched['HB_PLT_ratio'] = X_clinical['HB'] / (X_clinical['PLT'] + 1e-5)
    
    if 'WBC' in X_clinical.columns and 'MONOCYTES' in X_clinical.columns:
        X_enriched['WBC_MONOCYTES_ratio'] = X_clinical['WBC'] / (X_clinical['MONOCYTES'] + 1e-5)
    
    if 'BM_BLAST' in X_clinical.columns and 'WBC' in X_clinical.columns:
        X_enriched['BLAST_WBC_ratio'] = X_clinical['BM_BLAST'] / (X_clinical['WBC'] + 1e-5)
    
    # === 3. CLINICAL × CLINICAL INTERACTIONS ===
    # Most important clinical features based on literature
    important_pairs = [
        ('BM_BLAST', 'WBC'),
        ('BM_BLAST', 'HB'),
        ('BM_BLAST', 'PLT'),
        ('WBC', 'HB'),
        ('HB', 'PLT'),
        ('ANC', 'MONOCYTES')
    ]
    
    for col1, col2 in important_pairs:
        if col1 in X_clinical.columns and col2 in X_clinical.columns:
            X_enriched[f'{col1}_x_{col2}'] = X_clinical[col1] * X_clinical[col2]
    
    # === 4. MOLECULAR × CLINICAL INTERACTIONS ===
    if 'mutation_count_total' in X_molecular.columns:
        if 'BM_BLAST' in X_clinical.columns:
            X_enriched['mutations_x_BLAST'] = X_molecular['mutation_count_total'] * X_clinical['BM_BLAST']
        if 'WBC' in X_clinical.columns:
            X_enriched['mutations_x_WBC'] = X_molecular['mutation_count_total'] * X_clinical['WBC']
    
    if 'vaf_sum' in X_molecular.columns:
        if 'BM_BLAST' in X_clinical.columns:
            X_enriched['vaf_sum_x_BLAST'] = X_molecular['vaf_sum'] * X_clinical['BM_BLAST']
    
    # === 5. BINNING IMPORTANT CONTINUOUS VARIABLES ===
    # Create categorical bins for BM_BLAST (clinically meaningful thresholds)
    if 'BM_BLAST' in X_clinical.columns:
        X_enriched['BLAST_low'] = (X_clinical['BM_BLAST'] < 20).astype(int)
        X_enriched['BLAST_high'] = (X_clinical['BM_BLAST'] >= 50).astype(int)
    
    return X_enriched

print("✓ Enriched feature engineering function defined")

✓ Enriched feature engineering function defined


## 5. Build All Features

In [5]:
# Clean target
target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

# === TRAIN ===
# Clinical features
clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_clinical_train = clinical_train_clean[numeric_features].copy()
center_encoded_train = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical_train = pd.concat([X_clinical_train, center_encoded_train], axis=1)

# Cytogenetic + Molecular
cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train_clean.index.unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids, top_n_genes=20)

# Align molecular features
mol_features_train_aligned = mol_features_train.reindex(X_clinical_train.index, fill_value=0)

# === ADD ENRICHED FEATURES (TRAIN) ===
enriched_features_train = add_enriched_features(X_clinical_train[numeric_features], mol_features_train_aligned)
print(f"✓ Enriched features created (train): {enriched_features_train.shape[1]} new features")

# Combine: Clinical + Molecular + Cytogenetic + ENRICHED
cyto_features_train_aligned = cyto_features_train.reindex(X_clinical_train.index, fill_value=0)
X_combined_train = pd.concat([
    X_clinical_train, 
    mol_features_train_aligned, 
    cyto_features_train_aligned,
    enriched_features_train  # NEW!
], axis=1)

# === TEST ===
X_clinical_test = clinical_test.set_index('ID')[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test.set_index('ID')['CENTER'], prefix='CENTER', drop_first=True)
X_clinical_test = pd.concat([X_clinical_test, center_encoded_test], axis=1)

# Ensure same columns
for col in X_clinical_train.columns:
    if col not in X_clinical_test.columns:
        X_clinical_test[col] = 0
X_clinical_test = X_clinical_test[X_clinical_train.columns]

cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids, top_n_genes=20)

# Ensure same columns in molecular
for col in mol_features_train.columns:
    if col not in mol_features_test.columns:
        mol_features_test[col] = 0
mol_features_test = mol_features_test[mol_features_train.columns]

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)

# === ADD ENRICHED FEATURES (TEST) ===
enriched_features_test = add_enriched_features(X_clinical_test[numeric_features], mol_features_test_aligned)

cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)
X_combined_test = pd.concat([
    X_clinical_test, 
    mol_features_test_aligned, 
    cyto_features_test_aligned,
    enriched_features_test  # NEW!
], axis=1)

print(f"\n✓ Total features (with enrichment):")
print(f"  Train: {X_combined_train.shape}")
print(f"  Test: {X_combined_test.shape}")
print(f"  Improvement vs V21: +{enriched_features_train.shape[1]} features")

✓ Enriched features created (train): 27 new features

✓ Total features (with enrichment):
  Train: (3173, 132)
  Test: (1193, 132)
  Improvement vs V21: +27 features


## 6. Preprocessing

In [6]:
# Imputation
imputer = SimpleImputer(strategy='median')
X_imputed_train = pd.DataFrame(
    imputer.fit_transform(X_combined_train),
    index=X_combined_train.index,
    columns=X_combined_train.columns
)

X_imputed_test = pd.DataFrame(
    imputer.transform(X_combined_test),
    index=X_combined_test.index,
    columns=X_combined_test.columns
)

# Standardization
scaler = StandardScaler()
X_scaled_train = pd.DataFrame(
    scaler.fit_transform(X_imputed_train),
    index=X_imputed_train.index,
    columns=X_imputed_train.columns
)

X_scaled_test = pd.DataFrame(
    scaler.transform(X_imputed_test),
    index=X_imputed_test.index,
    columns=X_imputed_test.columns
)

# Create survival target
y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled_train, y_surv, test_size=0.2, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print(f"✓ Preprocessed data:")
print(f"  Train: {X_train.shape[0]} patients, {X_train.shape[1]} features")
print(f"  Val: {X_val.shape[0]} patients")

✓ Preprocessed data:
  Train: 2538 patients, 132 features
  Val: 635 patients


## 7. CoxNet with Extended Cross-Validation

In [7]:
print("="*60)
print("COXNET V21.1 - ENRICHED FEATURES")
print("="*60)
print("\nTesting multiple alpha AND l1_ratio values...\n")

# Test different configurations
alphas = [0.001, 0.01, 0.1, 1.0]
l1_ratios = [0.3, 0.5, 0.7, 0.9]  # Different L1/L2 mixes

results = []

for l1_ratio in l1_ratios:
    for alpha in alphas:
        coxnet = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alphas=[alpha],
            max_iter=100000,
            fit_baseline_model=True
        )
        
        coxnet.fit(X_train.values, y_train)
        
        y_pred_train = coxnet.predict(X_train.values)
        y_pred_val = coxnet.predict(X_val.values)
        
        c_index_train = concordance_index_censored(
            y_train['OS_STATUS'], y_train['OS_YEARS'], y_pred_train
        )[0]
        
        c_index_val = concordance_index_censored(
            y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_val
        )[0]
        
        n_nonzero = np.sum(coxnet.coef_.flatten() != 0)
        
        results.append({
            'l1_ratio': l1_ratio,
            'alpha': alpha,
            'c_index_train': c_index_train,
            'c_index_val': c_index_val,
            'overfitting': c_index_train - c_index_val,
            'n_features_selected': n_nonzero
        })
        
        print(f"L1_ratio: {l1_ratio:.1f} | Alpha: {alpha:6.3f} | "
              f"Train: {c_index_train:.4f} | Val: {c_index_val:.4f} | "
              f"Features: {n_nonzero}/{X_train.shape[1]}")

results_df = pd.DataFrame(results).sort_values('c_index_val', ascending=False)

print("\n" + "="*60)
print("TOP 10 CONFIGURATIONS")
print("="*60)
print(results_df.head(10).to_string(index=False))

# Select best
best_idx = results_df.index[0]
best_alpha = results_df.loc[best_idx, 'alpha']
best_l1_ratio = results_df.loc[best_idx, 'l1_ratio']
best_c_index = results_df.loc[best_idx, 'c_index_val']

print(f"\n✓ Best configuration:")
print(f"  L1_ratio: {best_l1_ratio}")
print(f"  Alpha: {best_alpha}")
print(f"  Validation C-index: {best_c_index:.4f}")
print(f"\n📊 Comparison with V21 baseline: improvement will be shown next...")

COXNET V21.1 - ENRICHED FEATURES

Testing multiple alpha AND l1_ratio values...

L1_ratio: 0.3 | Alpha:  0.001 | Train: 0.7541 | Val: 0.7335 | Features: 123/132
L1_ratio: 0.3 | Alpha:  0.010 | Train: 0.7524 | Val: 0.7374 | Features: 94/132
L1_ratio: 0.3 | Alpha:  0.100 | Train: 0.7425 | Val: 0.7339 | Features: 33/132
L1_ratio: 0.3 | Alpha:  1.000 | Train: 0.5000 | Val: 0.5000 | Features: 0/132
L1_ratio: 0.5 | Alpha:  0.001 | Train: 0.7542 | Val: 0.7340 | Features: 119/132
L1_ratio: 0.5 | Alpha:  0.010 | Train: 0.7515 | Val: 0.7370 | Features: 84/132
L1_ratio: 0.5 | Alpha:  0.100 | Train: 0.7384 | Val: 0.7325 | Features: 19/132
L1_ratio: 0.5 | Alpha:  1.000 | Train: 0.5000 | Val: 0.5000 | Features: 0/132
L1_ratio: 0.7 | Alpha:  0.001 | Train: 0.7542 | Val: 0.7341 | Features: 116/132
L1_ratio: 0.7 | Alpha:  0.010 | Train: 0.7507 | Val: 0.7369 | Features: 73/132
L1_ratio: 0.7 | Alpha:  0.100 | Train: 0.7318 | Val: 0.7292 | Features: 15/132
L1_ratio: 0.7 | Alpha:  1.000 | Train: 0.5000 | V

## 8. Train Final Model

In [8]:
print("\n" + "="*60)
print("TRAINING FINAL MODEL")
print("="*60)

coxnet_final = CoxnetSurvivalAnalysis(
    l1_ratio=best_l1_ratio,
    alphas=[best_alpha],
    max_iter=100000,
    fit_baseline_model=True
)

coxnet_final.fit(X_scaled_train.values, y_surv)

y_pred_val_final = coxnet_final.predict(X_val.values)
c_index_val_final = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_val_final
)[0]

print(f"\n✓ Final Model Performance:")
print(f"  Validation C-index: {c_index_val_final:.4f}")
print(f"\n📊 Comparison:")
print(f"  - RSF V2:              0.7334")
print(f"  - CoxNet V21:          ~0.73xx (baseline)")
print(f"  - CoxNet V21.1:        {c_index_val_final:.4f} ⭐")
print(f"\n  Features used: {np.sum(coxnet_final.coef_.flatten() != 0)}/{X_scaled_train.shape[1]}")


TRAINING FINAL MODEL

✓ Final Model Performance:
  Validation C-index: 0.7493

📊 Comparison:
  - RSF V2:              0.7334
  - CoxNet V21:          ~0.73xx (baseline)
  - CoxNet V21.1:        0.7493 ⭐

  Features used: 95/132


## 9. Feature Importance - Top Enriched Features

In [11]:
# Get all feature importance
feature_importance = pd.DataFrame({
    'feature': X_scaled_train.columns,
    'coefficient': coxnet_final.coef_.flatten()
})

feature_importance['abs_coef'] = np.abs(feature_importance['coefficient'])
feature_importance = feature_importance.sort_values('abs_coef', ascending=False)

# Identify enriched features
enriched_feature_names = enriched_features_train.columns.tolist()
feature_importance['is_enriched'] = feature_importance['feature'].isin(enriched_feature_names)

print("="*60)
print("TOP 20 FEATURES (overall)")
print("="*60)
top_20 = feature_importance.head(20).copy()
top_20['type'] = top_20['is_enriched'].map({True: '🆕 ENRICHED', False: 'BASE'})
print(top_20[['feature', 'coefficient', 'type']].to_string(index=False))

# Show top enriched features
enriched_only = feature_importance[feature_importance['is_enriched']].head(15)
print("\n" + "="*60)
print("TOP 15 ENRICHED FEATURES (interactions, transformations, ratios)")
print("="*60)
print(enriched_only[['feature', 'coefficient']].to_string(index=False))

n_enriched_selected = feature_importance[feature_importance['is_enriched'] & (feature_importance['coefficient'] != 0)].shape[0]
print(f"\n✨ Enriched features selected by model: {n_enriched_selected}/{len(enriched_feature_names)}")

TOP 20 FEATURES (overall)
             feature  coefficient       type
                  HB    -0.260854       BASE
mutation_count_total     0.161556       BASE
         cyto_normal    -0.159434       BASE
        BM_BLAST_log     0.156692 🆕 ENRICHED
             vaf_max     0.133723       BASE
     gene_TP53_count     0.128566       BASE
     vaf_sum_x_BLAST    -0.124856 🆕 ENRICHED
   mutations_x_BLAST    -0.123163 🆕 ENRICHED
      BM_BLAST_x_WBC     0.102524 🆕 ENRICHED
   gene_TP53_present     0.100121       BASE
                 PLT    -0.096039       BASE
             vaf_sum     0.095503       BASE
          effect_PTD     0.095435       BASE
  gene_ASXL1_present     0.090055       BASE
          CENTER_MUV     0.088429       BASE
        cyto_complex     0.086853       BASE
  gene_RUNX1_present     0.082912       BASE
            BM_BLAST     0.081129       BASE
            HB_x_PLT    -0.075414 🆕 ENRICHED
    gene_SRSF2_count     0.075391       BASE

TOP 15 ENRICHED FEATURES (in

In [13]:
# =============================================================================
# MODEL CONFIGURATION SUMMARY - CoxNet V21.1 Enriched
# =============================================================================

print("="*80)
print("MODEL CONFIGURATION SUMMARY - CoxNet V21.1 Enriched")
print("="*80)

# === 1. FEATURE ENGINEERING ===
print("\n" + "="*80)
print("1. FEATURE ENGINEERING")
print("="*80)

print(f"\n📊 Total features: {X_train.shape[1]}")
print(f"   Clinical features: {len([c for c in X_train.columns if c in numeric_features or c.startswith('CENTER')])}")
print(f"   Molecular features: {len([c for c in X_train.columns if c.startswith('gene_') or c.startswith('effect_') or c.startswith('vaf_') or c == 'mutation_count_total'])}")
print(f"   Cytogenetic features: {len([c for c in X_train.columns if c.startswith('cyto_')])}")
print(f"   Enriched features: {len([c for c in X_train.columns if '_log' in c or '_sqrt' in c or '_squared' in c or '_ratio' in c or '_x_' in c or 'BLAST_low' in c or 'BLAST_high' in c])}")

print("\n✨ Enriched feature breakdown:")
print(f"   - Non-linear transforms: {len([c for c in X_train.columns if '_log' in c or '_sqrt' in c or '_squared' in c])}")
print(f"     • log transforms: {len([c for c in X_train.columns if '_log' in c])}")
print(f"     • sqrt transforms: {len([c for c in X_train.columns if '_sqrt' in c])}")
print(f"     • squared transforms: {len([c for c in X_train.columns if '_squared' in c])}")
print(f"   - Clinical ratios: {len([c for c in X_train.columns if '_ratio' in c])}")
print(f"   - Clinical × Clinical interactions: {len([c for c in X_train.columns if '_x_' in c and not 'mutations_x' in c and not 'vaf_sum_x' in c])}")
print(f"   - Molecular × Clinical interactions: {len([c for c in X_train.columns if 'mutations_x' in c or 'vaf_sum_x' in c])}")
print(f"   - Binning features: {len([c for c in X_train.columns if 'BLAST_low' in c or 'BLAST_high' in c])}")

print("\n📋 Sample of enriched features:")
enriched_cols = [c for c in X_train.columns if '_log' in c or '_sqrt' in c or '_squared' in c or '_ratio' in c or '_x_' in c or 'BLAST_low' in c or 'BLAST_high' in c]
for i, feat in enumerate(enriched_cols[:15], 1):
    print(f"   {i:2d}. {feat}")
if len(enriched_cols) > 15:
    print(f"   ... and {len(enriched_cols) - 15} more enriched features")

# === 2. MODEL TYPE & ALGORITHM ===
print("\n" + "="*80)
print("2. MODEL TYPE & ALGORITHM")
print("="*80)

print(f"\n🔬 Model: Cox Elastic Net (CoxnetSurvivalAnalysis)")
print(f"   Family: Linear model for survival analysis")
print(f"   Method: Cox proportional hazards with regularization")
print(f"   Library: scikit-survival (sksurv.linear_model)")

print(f"\n📐 Regularization:")
print(f"   Type: Elastic Net (L1 + L2)")
print(f"   L1_ratio: {best_l1_ratio}")
print(f"     • L1 (Lasso) weight: {best_l1_ratio*100:.0f}%")
print(f"     • L2 (Ridge) weight: {(1-best_l1_ratio)*100:.0f}%")
print(f"   Alpha (strength): {best_alpha}")
print(f"   Max iterations: 100,000")
print(f"   Baseline model: Yes")

# === 3. HYPERPARAMETER SEARCH ===
print("\n" + "="*80)
print("3. HYPERPARAMETER SEARCH")
print("="*80)

print(f"\n🔍 Cross-validation configuration:")
print(f"   Alpha values tested: {alphas}")
print(f"   L1_ratio values tested: {l1_ratios}")
print(f"   Total configurations: {len(alphas) * len(l1_ratios)}")

print(f"\n🏆 Best configuration:")
print(f"   Best alpha: {best_alpha}")
print(f"   Best l1_ratio: {best_l1_ratio}")
print(f"   Selected based on: Validation C-index")

print(f"\n📊 Top 5 configurations by validation C-index:")
print(results_df.head(5)[['l1_ratio', 'alpha', 'c_index_val', 'n_features_selected']].to_string(index=False))

# === 4. FEATURE SELECTION ===
print("\n" + "="*80)
print("4. FEATURE SELECTION (via L1 penalty)")
print("="*80)

n_selected = np.sum(coxnet_final.coef_.flatten() != 0)
n_total = len(coxnet_final.coef_.flatten())

print(f"\n🎯 Features selected: {n_selected} / {n_total} ({n_selected/n_total*100:.1f}%)")
print(f"   Features with zero coefficient: {n_total - n_selected}")
print(f"   Sparsity achieved: {(1 - n_selected/n_total)*100:.1f}%")

# === 5. DATA SPLIT ===
print("\n" + "="*80)
print("5. DATA SPLIT")
print("="*80)

print(f"\n📦 Dataset sizes:")
print(f"   Total patients: {len(y_surv)}")
print(f"   Train: {X_train.shape[0]} patients ({X_train.shape[0]/len(y_surv)*100:.1f}%)")
print(f"   Validation: {X_val.shape[0]} patients ({X_val.shape[0]/len(y_surv)*100:.1f}%)")
print(f"   Test: {X_scaled_test.shape[0]} patients")

print(f"\n⚖️  Stratification: By OS_STATUS (event/censored)")
print(f"   Events in dataset: {y_surv['OS_STATUS'].sum()} / {len(y_surv)} ({y_surv['OS_STATUS'].sum()/len(y_surv)*100:.1f}%)")

# === 6. PREPROCESSING ===
print("\n" + "="*80)
print("6. PREPROCESSING")
print("="*80)

print(f"\n🔧 Imputation:")
print(f"   Strategy: median")
print(f"   Library: SimpleImputer (sklearn)")

print(f"\n📊 Scaling:")
print(f"   Method: StandardScaler (z-score normalization)")
print(f"   Library: sklearn")
print(f"   Importance: CRITICAL for penalized models")
print(f"   Applied to: All {X_train.shape[1]} features")

# === 7. TOP FEATURES (by absolute coefficient) ===
print("\n" + "="*80)
print("7. TOP 15 MOST IMPORTANT FEATURES")
print("="*80)

print("\n(Ranked by absolute coefficient value)")
print(feature_importance[['feature', 'coefficient']].head(15).to_string(index=False))

# Count enriched features in top 15
top15_features = feature_importance.head(15)['feature'].tolist()
n_enriched_top15 = len([f for f in top15_features if '_log' in f or '_sqrt' in f or '_squared' in f or '_ratio' in f or '_x_' in f or 'BLAST_low' in f or 'BLAST_high' in f])

print(f"\n✨ Enriched features in top 15: {n_enriched_top15}/15 ({n_enriched_top15/15*100:.0f}%)")

# === 8. PERFORMANCE METRICS ===
print("\n" + "="*80)
print("8. PERFORMANCE METRICS")
print("="*80)

print(f"\n🎯 Final model performance:")
print(f"   Validation C-index: {c_index_val_final:.4f}")

print(f"\n📈 Comparison with baselines:")
print(f"   RSF V2:              0.7334")
print(f"   CoxNet V21 (base):   ~0.73xx")
print(f"   CoxNet V21.1:        {c_index_val_final:.4f} ⭐")

print(f"\n🔄 Training vs Validation (on best config):")
best_config = results_df.iloc[0]
print(f"   Train C-index: {best_config['c_index_train']:.4f}")
print(f"   Val C-index: {best_config['c_index_val']:.4f}")
print(f"   Overfitting gap: {best_config['overfitting']:.4f}")

# === 9. MODEL COEFFICIENTS STATISTICS ===
print("\n" + "="*80)
print("9. COEFFICIENT STATISTICS")
print("="*80)

coefs = coxnet_final.coef_.flatten()
nonzero_coefs = coefs[coefs != 0]

print(f"\n📊 Coefficient distribution:")
print(f"   Min coefficient: {nonzero_coefs.min():.4f}")
print(f"   Max coefficient: {nonzero_coefs.max():.4f}")
print(f"   Mean |coefficient|: {np.abs(nonzero_coefs).mean():.4f}")
print(f"   Median |coefficient|: {np.median(np.abs(nonzero_coefs)):.4f}")

print(f"\n⚖️  Sign distribution:")
positive_coefs = np.sum(coefs > 0)
negative_coefs = np.sum(coefs < 0)
print(f"   Positive (risk ↑): {positive_coefs} ({positive_coefs/n_selected*100:.1f}%)")
print(f"   Negative (risk ↓): {negative_coefs} ({negative_coefs/n_selected*100:.1f}%)")

# === 10. SUBMISSION INFO ===
print("\n" + "="*80)
print("10. SUBMISSION DETAILS")
print("="*80)

print(f"\n📁 Output file: submission_v21.1_coxnet_enriched.csv")
print(f"   Predictions: {X_scaled_test.shape[0]} patients")
"""print(f"   Risk score range: [{submission['OS_YEARS'].min():.4f}, {submission['OS_YEARS'].max():.4f}]")
print(f"   Risk score mean: {submission['OS_YEARS'].mean():.4f}")
print(f"   Risk score std: {submission['OS_YEARS'].std():.4f}")"""

print("\n" + "="*80)
print("END OF CONFIGURATION SUMMARY")
print("="*80 + "\n")

MODEL CONFIGURATION SUMMARY - CoxNet V21.1 Enriched

1. FEATURE ENGINEERING

📊 Total features: 132
   Clinical features: 28
   Molecular features: 61
   Cytogenetic features: 17
   Enriched features: 27

✨ Enriched feature breakdown:
   - Non-linear transforms: 12
     • log transforms: 4
     • sqrt transforms: 4
     • squared transforms: 4
   - Clinical ratios: 4
   - Clinical × Clinical interactions: 6
   - Molecular × Clinical interactions: 3
   - Binning features: 2

📋 Sample of enriched features:
    1. BM_BLAST_log
    2. BM_BLAST_sqrt
    3. BM_BLAST_squared
    4. WBC_log
    5. WBC_sqrt
    6. WBC_squared
    7. ANC_log
    8. ANC_sqrt
    9. ANC_squared
   10. MONOCYTES_log
   11. MONOCYTES_sqrt
   12. MONOCYTES_squared
   13. WBC_ANC_ratio
   14. HB_PLT_ratio
   15. WBC_MONOCYTES_ratio
   ... and 12 more enriched features

2. MODEL TYPE & ALGORITHM

🔬 Model: Cox Elastic Net (CoxnetSurvivalAnalysis)
   Family: Linear model for survival analysis
   Method: Cox proportional h

Model save


In [10]:
"""
Script to save CoxNet V21.1 model artifacts for use in ensemble
Add this code at the end of notebook 21.1_CoxNet_Enriched.ipynb
"""
import pickle
import os

# After training coxnet_final in notebook 21.1, run this:
def save_coxnet_model(coxnet_final, imputer, scaler, X_combined_train, 
                      best_l1_ratio, best_alpha, c_index_val_final, DATA_PATH):
    """
    Save CoxNet model and all preprocessing artifacts
    
    Parameters:
    -----------
    coxnet_final : CoxnetSurvivalAnalysis
        Trained CoxNet model
    imputer : SimpleImputer
        Fitted imputer
    scaler : StandardScaler
        Fitted scaler
    X_combined_train : pd.DataFrame
        Training features (to save column names)
    best_l1_ratio : float
        Best l1_ratio parameter
    best_alpha : float
        Best alpha parameter
    c_index_val_final : float
        Validation C-index
    DATA_PATH : str
        Path to save the model
    """
    
    model_artifacts = {
        'model': coxnet_final,
        'imputer': imputer,
        'scaler': scaler,
        'feature_columns': X_combined_train.columns.tolist(),
        'best_params': {
            'l1_ratio': best_l1_ratio,
            'alpha': best_alpha
        },
        'performance': {
            'c_index_val': c_index_val_final
        }
    }
    
    model_path = os.path.join(DATA_PATH, 'coxnet_v21.1_model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model_artifacts, f)
    
    print("=" * 60)
    print("MODEL SAVED")
    print("=" * 60)
    print(f"✓ File: {model_path}")
    print(f"✓ Model: CoxNet V21.1")
    print(f"✓ C-index: {c_index_val_final:.4f}")
    print(f"✓ Features: {len(X_combined_train.columns)}")
    print(f"\nThis model can now be loaded in the ensemble notebook (V22.1)")
    
    return model_path



import pickle

# Save the final model and preprocessors
model_artifacts = {
    'model': coxnet_final,
    'imputer': imputer,
    'scaler': scaler,
    'feature_columns': X_combined_train.columns.tolist(),
    'best_params': {
        'l1_ratio': best_l1_ratio,
        'alpha': best_alpha
    },
    'performance': {
        'c_index_val': c_index_val_final
    }
}

model_path = os.path.join(DATA_PATH, 'coxnet_v21.1_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_artifacts, f)

print("=" * 60)
print("MODEL SAVED")
print("=" * 60)
print(f"✓ File: {model_path}")
print(f"✓ Model: CoxNet V21.1")
print(f"✓ C-index: {c_index_val_final:.4f}")
print(f"✓ Features: {len(X_combined_train.columns)}")
print(f"\nThis model can now be loaded in the ensemble notebook (V22.1)")



MODEL SAVED
✓ File: C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie\coxnet_v21.1_model.pkl
✓ Model: CoxNet V21.1
✓ C-index: 0.7493
✓ Features: 132

This model can now be loaded in the ensemble notebook (V22.1)


## 10. Predictions & Submission

In [ ]:
# Generate predictions
test_risk_scores = coxnet_final.predict(X_scaled_test.values)

# Create submission
submission = pd.DataFrame({
    'ID': X_scaled_test.index,
    'OS_YEARS': test_risk_scores
})

# Save submission
submission_path = os.path.join(DATA_PATH, "submission_v21.1_coxnet_enriched.csv")
submission.to_csv(submission_path, index=False)

print("="*60)
print("SUBMISSION CREATED")
print("="*60)
print(f"✓ File: {submission_path}")
print(f"\nPreview:")
print(submission.head(10))
print(f"\nRisk score statistics:")
print(submission['OS_YEARS'].describe())

## 11. Summary

In [ ]:
print("="*60)
print("SUMMARY - CoxNet V21.1 (Enriched Features)")
print("="*60)
print(f"\n🎯 Performance:")
print(f"  Final Validation C-index: {c_index_val_final:.4f}")
print(f"\n✨ Enriched Features Added: {len(enriched_feature_names)}")
print(f"  - Non-linear transforms: log, sqrt, squared")
print(f"  - Clinical ratios: WBC/ANC, HB/PLT, etc.")
print(f"  - Clinical interactions: BM_BLAST×WBC, etc.")
print(f"  - Molecular×Clinical: mutations×BLAST, etc.")
print(f"\n📊 Model Configuration:")
print(f"  Best L1_ratio: {best_l1_ratio}")
print(f"  Best alpha: {best_alpha}")
print(f"  Features selected: {np.sum(coxnet_final.coef_.flatten() != 0)}/{X_scaled_train.shape[1]}")
print(f"  Enriched features used: {n_enriched_selected}")
print(f"\n💡 Next Steps:")
print(f"  - Ready for ensemble (RSF + NN + CoxNet V21.1)")
print(f"  - Can submit if satisfied with performance")
print(f"  - Analyze which enriched features are most useful")
print("="*60)